#### Теория: Курсор

С результатом запроса мы взаимодействуем через объект `cursor`, который можно получить через метод `cursor()` объекта соединения.

##### `получение объекта курсора`

```python
cursor = conn.cursor()
```

Для получения результата после выполнения запроса используются следующие команды:

- `cursor.fetchone()` — вернуть одну строку

- `cursor.fetchall()` — вернуть все строки

- `cursor.fetchmany(size=10)` — вернуть указанное количество строк

```python
curs.execute("SELECT * FROM users;")
all_users = curs.fetchall()
```

Также курсоры итерируемы, так что получить результаты запросов можно обходом.

```python
curs.execute("SELECT * FROM users;")
for row in curs:
    print(row)
```

- Курсоры привязаны к соединению на весь срок жизни, и все команды выполняются в контексте одной сессии базы данных, обернутой соединением. 

- Любые изменения, сделанные в базе данных одним курсором, немедленно видны другим курсорам. Потому хорошей практикой при работе с базой данных является закрытие курсора и соединения с базой. Для автоматизации этого процесса удобно взаимодействовать через контекстный менеджер, используя конструкцию  with  :

```python
with conn.cursor() as curs:
    curs.execute("SELECT * FROM users;")
    all_users = curs.fetchall()
```

##### `Фабрика строк`

По умолчанию результат возвращается в виде кортежа. Такое поведение возможно изменить, передав параметр `row_factory` в момент открытия объекта `cursor`. Фабрики строк определены в модуле `psycopg.rows`.

Наиболее полезные фабрики:

- `dict_row` - возвращает данные в виде словаря

- `namedtuple_row` - возвращает данные в виде именованного кортежа, более легковесной альтернативы словарю


```python
from psycopg.rows import namedtuple_row

with conn.cursor(row_factory=namedtuple_row) as curs:
    curs.execute("SELECT * FROM users WHERE name=%s;", ("Alfred",))
    alfred = curs.fetchone()
    alfred  # Row(id=10, name='Alfred', age='90'
```
---

`Дополнение`

Фабрика строк — это функция, которая определяет, в каком виде ваша программа получит каждую строку из результата запроса.

`Как это работает`

Когда курсор выполняет `SELECT`, база возвращает данные в виде набора значений. 
`psycopg` должен решить:

- дефолт — кортеж `(10, 'Alfred', '90')`
- словарь `{'id': 10, 'name': 'Alfred', 'age': '90'}`
- именованный кортеж `Row(id=10, name='Alfred', age='90')`

Вы выбираете это поведение, передав `row_factory` при создании курсора. Это и есть «фабрика»: она фабрикует (создаёт) объекты-строки нужного типа из сырых данных БД.

`Cравнение`

1. Кортеж (по умолчанию) — быстро, но доступ по индексу:

```python
row = curs.fetchone()
row[1]   # 'Alfred' — непонятно, что тут, пока не запомнишь порядок колонок
```

2. `dict_row` — читается, доступ по имени:

```python
row['name']   # 'Alfred'
```
- Словарь хранит сами строковые ключи в каждой записи (тратит память)

3.`namedtuple_row` — золотая середина: и по имени можно, и доступ через точку, и память экономится по сравнению со словарём:

```python
row.name   # 'Alfred'
row.id     # 10
```
- именнованный кортеж хранит ключи один раз на уровне класса, поэтому легче, особенно при тысячах строк.


`Kак ещё можно указать фабрику`

Кроме `cursor`, фабрику можно задать на уровне соединения, чтобы все его курсоры наследовали её:

```python
conn.row_factory = dict_row
```

`Короткий итог`

Фабрика строк = настройка «в каком контейнере отдавать каждую строку результатов» (кортеж / словарь / именованный кортеж). Выбирается через cursor(row_factory=...) или conn.row_factory, модуль готовых фабрик — psycopg.rows.

`Mодуль готовых фабрик — psycopg.rows.`

В модуле psycopg.rows (psycopg 3) определены такие фабрики строк:

|Фабрика|	Что возвращает|	Доступ к полю|
|:------------------|:-----------------|:--------------|
|`tuple_row` (по умолчанию)|	обычный кортеж|	`row[1]`|
|`dict_row`|	словарь|	`row['name']`|
|`namedtuple_row`|	именованный кортеж| `Row(...)	row.name`|
|`class_row(класс)`|	экземпляр произвольного класса|	`row.name`|

Плюс в модуле есть декоратор `@row_factory`, которым можно пометить собственную фабрику — функцию вида `factory(cursor, values)` -> что_угодно.

`class_row` — отдельно

Самый гибкий вариант: передаёте ей ваш класс, и `psycopg` создаёт объекты этого класса, сопоставляя колонки с атрибутами/аргументами конструктора:

```python
from psycopg.rows import class_row

class User:
    def __init__(self, id, name, age):
        self.id, self.name, self.age = id, name, age

with conn.cursor(row_factory=class_row(User)) as curs:
    curs.execute("SELECT id, name, age FROM users WHERE name=%s;", ("Alfred",))
    alfred = curs.fetchone()   # объект User
    alfred.name                # 'Alfred'
```

##### Декоратор `@row_factory`

```python
from psycopg.rows import row_factory

@row_factory
def my_row(cursor, values):
    return my_own_container(values)
```

---

##### `Серверные курсоры`

Когда выполняется запрос к базе данных, курсор получает все записи, возвращаемые базой, бэкендом, передавая их в клиентский процесс. 
Если запрос возвращает большое количество данных, то и будет выделен пропорционально большой объем памяти на стороне клиента.

Если набор данных слишком велик для практической обработки на стороне клиента, то можно создать курсор на стороне сервера. 
Используя такой курсор, можно передавать клиенту только контролируемое количество данных, не храня весь объем полностью в памяти.

В `Psycopg` серверные курсоры называются `именованными курсорами`. 
Именованный курсор создается с помощью метода `cursor()` с указанием имени в первом аргументе. Такой курсор ведет себя как обычный курсор, позволяя пользователю перемещаться по набору данных с помощью метода `scroll()` и читать данные с помощью методов `fetchone()` и `fetchmany()`. По умолчанию можно перемещаться только вперед, но если вам нужно перемещаться назад, нужно объявить ваш курсор `scrollable`.

```python
with conn.cursor("cursor_name", scrollable=True) as curs:
    curs.execute("SELECT * FROM users;")
    result = curs.fetchall()
```

[`Самостоятельная работа`](sq04_1.ipynb)

- Выполните шаги из урока у себя на компьютере.

- Попробуйте использовать `namedtuple_row` из `psycopg.rows`.

- Подготовьте запрос, содержащий агрегирующую функцию, например такой:

```SELECT MAX(id) from users;```

- Выполните его и получите результат из именованного кортежа.`